# 02 - Clean the data
Turns the raw WFP file into a tidy table for analysis, and builds a WFP-to-FAOSTAT crop mapping.

Decisions made here (all based on `01_explore`):
- Drop the incomplete final month and the `National Average` pseudo-market.
- Keep only food crops with enough history (plus three short-history crops flagged as *indicative*).
- Leave `Maize (new harvest)`, `Maize (white)` and `Maize (seed)` out. They are separate series and would double count.
- Save to `clean_data/` and load into a DuckDB database.

In [2]:
import pandas as pd
import duckdb
from pathlib import Path

RAW, CLEAN = Path("..") / "raw_data", Path("..") / "clean_data"
CLEAN.mkdir(exist_ok=True)
wfp = pd.read_csv(RAW / "wfp_food_prices_mwi.csv", parse_dates=["date"])
print("raw rows:", len(wfp))

FileNotFoundError: [Errno 2] No such file or directory: '..\\raw_data\\wfp_food_prices_mwi.csv'

In [ ]:
# 1. drop the final month if it is incomplete (under half the rows of the month before it)
per_month = wfp.groupby("date").size()
last, prev = per_month.index[-1], per_month.index[-2]
if per_month[last] < 0.5 * per_month[prev]:
    print("dropping incomplete final month:", last.strftime("%Y-%m"), per_month[last], "rows vs", per_month[prev])
    wfp = wfp[wfp["date"] != last]

# 2. drop the national average pseudo-market
wfp = wfp[wfp["market"] != "National Average"]

In [ ]:
CROPS = {
    # WFP name              : (FAOSTAT item,       history tier)
    "Maize":                 ("Maize (corn)",         "strong"),
    "Rice":                  ("Rice",                 "strong"),
    "Cassava":               ("Cassava, fresh",       "strong"),
    "Beans":                 ("Beans, dry",           "strong"),
    "Groundnuts (shelled)":  ("Groundnuts, excluding shelled", "moderate"),
    "Cowpeas":               ("Cow peas, dry",        "indicative"),
    "Pigeon peas":           ("Pigeon peas, dry",     "indicative"),
    "Sweet potatoes":        ("Sweet potatoes",       "indicative"),
    "Sorghum":               ("Sorghum",              "indicative"),
    "Millet":                (None,                   "indicative"),   # not in the FAOSTAT extract
}
crop_map = (pd.DataFrame(CROPS, index=["fao_item", "history_tier"]).T
              .rename_axis("wfp_commodity").reset_index())
crop_map.to_csv(CLEAN / "crop_mapping.csv", index=False)
crop_map

In [ ]:
region = (wfp[["market", "admin1"]].dropna().drop_duplicates("market")
              .rename(columns={"admin1": "region"}))

prices = (wfp[wfp["commodity"].isin(CROPS)]
          .merge(region, on="market", how="left")
          [["date", "commodity", "unit", "market", "region", "price", "usdprice", "priceflag"]]
          .sort_values(["commodity", "market", "date"])
          .reset_index(drop=True))
prices["year"], prices["month"] = prices["date"].dt.year, prices["date"].dt.month

# every kept crop should be priced per KG
print(prices.groupby("commodity")["unit"].unique())
print(prices.shape)

In [ ]:
# one row per crop, market and month (a few markets report twice in a month)
before = len(prices)
prices = (prices.groupby(["commodity", "market", "region", "year", "month"], as_index=False)
                .agg(price=("price", "median"), usdprice=("usdprice", "median")))
prices["date"] = pd.to_datetime(dict(year=prices.year, month=prices.month, day=1))
print("collapsed", before, "->", len(prices), "rows")
prices.to_csv(CLEAN / "prices_clean.csv", index=False)

In [ ]:
con = duckdb.connect(str(CLEAN / "malawi.duckdb"))
con.execute("CREATE OR REPLACE TABLE prices_clean AS SELECT * FROM prices")
con.execute("CREATE OR REPLACE TABLE crop_mapping AS SELECT * FROM crop_map")
con.sql("SELECT commodity, COUNT(*) AS rows, COUNT(DISTINCT market) AS markets, "
        "MIN(year) AS first_year, MAX(year) AS last_year FROM prices_clean GROUP BY 1 ORDER BY rows DESC").show()
con.close()

**Next:** `03_seasonality.ipynb` reads `clean_data/prices_clean.csv` and builds the monthly seasonal index.